# Transcribe — Kaggle runner

**Before running:** accept the gate at https://huggingface.co/bodhan-ai/indic-transcribe-core, add an `HF_TOKEN` Kaggle Secret, turn **Internet On**, and choose a **GPU** accelerator. Kaggle documents these under Settings / Session options [1](https://www.kaggle.com/docs/notebooks).

Run the **GPU diagnostic** cell first. If `torch.cuda.is_available()` is `False`, fix the accelerator/session before running the **main** cell; the main cell now refuses to continue on CPU.

The main cell clones the branch, installs dependencies while pinning Kaggle's preinstalled CUDA `torch`/`torchaudio`, checks access, uses the bundled MP3 + Scribe 2 JSON, runs `pipeline.py`, and writes outputs to `/kaggle/working/transcribe_outputs/`.

If you want your own MP3, add it with **Add Input** and set `AUDIO_PATH` at the top of the main cell. Scribe 2 verification is skipped for different audio.


In [ ]:
# GPU diagnostic — RUN THIS CELL FIRST
import os
import subprocess
import importlib.metadata as md

print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
try:
    smi = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        text=True, stderr=subprocess.STDOUT,
    )
    print("nvidia-smi:\n" + smi.strip())
except Exception as exc:
    print("nvidia-smi FAILED:", exc)

try:
    import torch
    import torchaudio
    print("torch package:", md.version("torch"))
    print("torch.version.cuda:", torch.version.cuda)
    print("torchaudio package:", md.version("torchaudio"))
    print("torch.cuda.is_available():", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("Torch diagnostic FAILED:", exc)

print("\nIf nvidia-smi failed or torch.cuda.is_available() is False:")
print("  1. Kaggle Settings -> Accelerator -> GPU T4/P100")
print("  2. Settings -> Internet -> On")
print("  3. RESTART the session, then re-run this diagnostic")
print("  Docs: https://www.kaggle.com/docs/notebooks")


In [ ]:
# ============================================================
# MAIN KAGGLE CELL — run the GPU diagnostic first; edit only this block
# ============================================================
import os
import shutil
import subprocess
import sys
from pathlib import Path

BRANCH = "arena/793a5a3b-transcribe"
REPO_URL = "https://github.com/kethan/transcribe.git"
WORK = Path("/kaggle/working")
REPO = WORK / "transcribe"
OUT = WORK / "transcribe_outputs"

ASR_LANGUAGE = "hi"          # bundled MP3 is Hindi-heavy; use "gu" for Gujarati
ALIGNMENT_LANGUAGE = "hin"   # ISO 639-3 for uroman; use "guj" for Gujarati
AUDIO_PATH = ""              # blank = bundled MP3; or /kaggle/input/.../file.mp3

# Keep the big model caches off the 20 GB auto-saved /kaggle/working disk.
os.environ.setdefault("HF_HOME", "/kaggle/temp/hf_cache")
Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)

# The cudart_stub warning comes from TensorFlow; this pipeline uses PyTorch.
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("TRANSFORMERS_NO_TF", "1")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

# 1. Load HF_TOKEN from Kaggle Secrets. Never paste the token into this cell.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    if not os.environ.get("HF_TOKEN"):
        raise RuntimeError(
            "Add-ons -> Secrets -> add a secret named HF_TOKEN, enable it for "
            "this notebook, then re-run this cell. Accept the Bodhan gate first."
        ) from exc
print("HF_TOKEN loaded from Kaggle Secret or environment.")

# 2. Clone the exact branch. All those confusing files come along automatically.
if not REPO.exists():
    subprocess.check_call(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO)]
    )
else:
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH])
os.chdir(REPO)
print("Repository:", REPO)

# 3. Install dependencies, but pin Kaggle's preinstalled CUDA torch/torchaudio
# so pip cannot silently replace them with CPU wheels.
print("Installing dependencies ...", flush=True)
constraint_path = WORK / "torch_constraints.txt"
try:
    probe = subprocess.check_output(
        [sys.executable, "-c",
         "import importlib.metadata as m; print(m.version('torch')); print(m.version('torchaudio'))"],
        text=True,
    ).splitlines()
    if len(probe) == 2:
        constraint_path.write_text(
            f"torch=={probe[0]}\ntorchaudio=={probe[1]}\n", encoding="utf-8"
        )
        print(f"Pinning CUDA torch/torchaudio: torch=={probe[0]}, torchaudio=={probe[1]}")
    else:
        constraint_path.unlink(missing_ok=True)
except Exception as exc:
    print(f"WARNING: could not pin preinstalled torch/torchaudio: {exc}")
    constraint_path.unlink(missing_ok=True)

pip_cmd = [sys.executable, "-m", "pip", "install", "-q"]
if constraint_path.exists():
    pip_cmd += ["-c", str(constraint_path)]
pip_cmd += ["-r", "requirements.txt", "-r", "tools/requirements.txt"]
subprocess.check_call(pip_cmd)

# 4. Fail early if the GPU is not visible. The cudart_stub warning can come
# from TensorFlow; torch.cuda.is_available() is the check that matters here.
try:
    nvidia = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        text=True, stderr=subprocess.STDOUT,
    ).strip()
    print("nvidia-smi:", nvidia)
except Exception as exc:
    raise RuntimeError(
        "No NVIDIA driver is visible. Kaggle Settings -> Accelerator -> GPU, "
        "Internet On, then RESTART the session and re-run this notebook."
    ) from exc
try:
    import torch
    print(f"torch={torch.__version__}, torch.version.cuda={torch.version.cuda}, "
          f"CUDA available={torch.cuda.is_available()}")
    if not torch.cuda.is_available():
        raise RuntimeError(
            "PyTorch cannot see CUDA. Run the GPU diagnostic cell; either the "
            "GPU accelerator is off or pip replaced Kaggle's CUDA torch with a "
            "CPU build. Restart the session before continuing."
        )
except RuntimeError:
    raise
except Exception as exc:
    raise RuntimeError(f"PyTorch is not importable after install: {exc}") from exc
print("ffmpeg:", shutil.which("ffmpeg") or "MISSING")
OUT.mkdir(parents=True, exist_ok=True)

# 5. Confirm the gate was accepted with the same account as HF_TOKEN.
print("Checking Hugging Face access (small files only) ...", flush=True)
subprocess.check_call([sys.executable, "-m", "tools.check_hf_access"])

# 6. Find the audio and the ElevenLabs Scribe 2 reference.
bundled = REPO / "audio/20261031 CHD 3 GTH 11 12 13 14 15 (51) CHN.mp3"
reference = REPO / "audio/20261031 CHD 3 GTH 11 12 13 14 15 (51) CHN.json"
if AUDIO_PATH:
    audio = Path(AUDIO_PATH).expanduser().resolve()
elif bundled.exists():
    audio = bundled
else:
    candidates = sorted(Path("/kaggle/input").rglob("*.mp3"))
    if not candidates:
        raise FileNotFoundError("No MP3 found. Add one with Add Input or set AUDIO_PATH.")
    audio = max(candidates, key=lambda p: p.stat().st_size)
if not audio.exists():
    raise FileNotFoundError(audio)
if audio.suffix.lower() in {".mp3", ".m4a", ".aac", ".flac", ".ogg"} and not shutil.which("ffmpeg"):
    raise RuntimeError("ffmpeg is required to canonicalize non-WAV input.")
verify_reference = audio.name == bundled.name and reference.exists()
print(f"Audio: {audio}")
print(f"Scribe 2 verification: {'yes' if verify_reference else 'no (different audio)'}")

# 7. Convert the attached Scribe 2 JSON to the strict reference schema.
if verify_reference:
    subprocess.check_call([
        sys.executable, "-m", "tools.scribe2_reference", "convert",
        "--audio", str(audio),
        "--reference", str(reference),
        "--output", str(OUT / "scribe2_reference_strict.json"),
        "--report-json", str(OUT / "scribe2_conversion_report.json"),
        "--report-md", str(OUT / "scribe2_conversion_report.md"),
    ])

# 8. Run the real local-HF / NeMo / CTC pipeline.
cmd = [
    sys.executable, "pipeline.py",
    "--audio", str(audio),
    "--language", ASR_LANGUAGE,
    "--alignment-language", ALIGNMENT_LANGUAGE,
    "--output-dir", str(OUT),
]
print("Running pipeline:", " ".join(cmd), flush=True)
subprocess.check_call(cmd)

# 9. Compare the real transcript with the Scribe 2 reference.
if verify_reference:
    subprocess.check_call([
        sys.executable, "-m", "tools.scribe2_reference", "verify",
        "--candidate", str(OUT / "transcript.json"),
        "--audio", str(audio),
        "--reference", str(reference),
        "--report-json", str(OUT / "scribe2_verification_report.json"),
        "--report-md", str(OUT / "scribe2_verification_report.md"),
    ])

print("\nDONE. Download these files before stopping the notebook:")
for p in sorted(OUT.glob("*")):
    if p.is_file():
        print(f" - {p}  ({p.stat().st_size:,} bytes)")
